# 1. Spark Application Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("Assignment_6").master("local[*]").getOrCreate()

# 2. Data Ingestion & Validation

In [3]:
products_schema = StructType([
    StructField("product_id",StringType(),False),
    StructField("product_name",StringType(),False),
    StructField("brand",StringType(),False)
])

transactions_schema = StructType([
    StructField("transaction_id",StringType(),False),
    StructField("user_id",StringType(),False),
    StructField("product_id",StringType(),False),
    StructField("category",StringType(),False),
    StructField("price",FloatType(),False),
    StructField("quantity",IntegerType(),False),
    StructField("transaction_timestamp",TimestampType(),False),
    StructField("platform",StringType(),False),
    StructField("country",StringType(),False)
])

In [4]:
products = spark.read\
                .format("csv")\
                .option("header",True)\
                .load(r"C:\Users\aman.rajput\Downloads\Assignment_6\products.csv")

transactions = spark.read\
                    .format("csv")\
                    .option("header",True)\
                    .load(r"C:\Users\aman.rajput\Downloads\Assignment_6\transactions.csv")

**Seperating valid records from invalid records**:

In [5]:
valid_products = products.dropna(how='any')
valid_transactions = transactions.filter(col("transaction_timestamp") != 'invalid_timestamp')\
                                .filter(col("price").rlike("^[0-9]+(\\.[0-9]+)?$"))\
                                .filter(col("quantity").rlike("[0-9]+"))\
                                .dropna(how='any')

In [6]:
invalid_transactions = transactions.exceptAll(valid_transactions)

In [7]:
invalid_transactions.count()

16

In [8]:
invalid_transactions.show()

+--------------+--------+----------+-------------+------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|     category| price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-------------+------+--------+---------------------+--------+-------+
|        TXN205|    NULL|   PROD005|        Books| 25.99|       1|  2024-02-04 12:30:45|  Mobile|     UK|
|        TXN043|USER1043|   PROD043|      Fashion|  NULL|       2|  2024-01-19 10:22:15|  Mobile|Germany|
|        TXN206|USER1206|   PROD006|  ELECTRONICS|   abc|       1|  2024-02-04 13:15:20|     Web|    USA|
|        TXN008|USER1008|      NULL|  Electronics|399.99|       1|  2024-01-15 17:05:30|     Web| Canada|
|          NULL|    NULL|      NULL|         NULL|  NULL|    NULL|                 NULL|    NULL|   NULL|
|        TXN009|    NULL|   PROD009|        Books| 15.99|       1|  2024-01-15 18:12:22|  Mobile|     UK|
|        TXN005|USER1005|   PROD005|        Bo

**Audit: Report the total number of records vs. incomplete records.**

In [9]:
print(f"Products total records = {products.count()} | Products Incomplete records = {products.count() - valid_products.count()}")
print(f"Transactions total records = {transactions.count()} | Transactions Incomplete records = {transactions.count() - valid_transactions.count()}")

Products total records = 100 | Products Incomplete records = 0
Transactions total records = 214 | Transactions Incomplete records = 16



**Schema: Enforce an explicit schema (StructType) during the ingestion of valid data.**

In [10]:
# Fix for transactions_df locally to cast string columns to correct types
transactions_cleaned = valid_transactions.withColumn("price", col("price").cast(FloatType())) \
                                          .withColumn("quantity", col("quantity").cast(IntegerType())) \
                                          .withColumn("transaction_timestamp", expr("try_cast(transaction_timestamp AS timestamp)"))\
                                          .dropna(how='any')

In [11]:
transactions_cleaned.filter(col("transaction_timestamp").isNull()).show()

+--------------+-------+----------+--------+-----+--------+---------------------+--------+-------+
|transaction_id|user_id|product_id|category|price|quantity|transaction_timestamp|platform|country|
+--------------+-------+----------+--------+-----+--------+---------------------+--------+-------+
+--------------+-------+----------+--------+-----+--------+---------------------+--------+-------+



In [12]:
products_df = spark.createDataFrame(valid_products.rdd,schema=products_schema)
transactions_df = spark.createDataFrame(transactions_cleaned.rdd,schema=transactions_schema)

In [13]:
print(products_df.printSchema())
print(transactions_df.printSchema())

root
 |-- product_id: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- brand: string (nullable = false)

None
root
 |-- transaction_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- product_id: string (nullable = false)
 |-- category: string (nullable = false)
 |-- price: float (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- transaction_timestamp: timestamp (nullable = false)
 |-- platform: string (nullable = false)
 |-- country: string (nullable = false)

None


# 3. Data Cleaning & Standardization

**Null Handling: Remove or impute missing values.**<br>
I have decided to remove the missing values using the dropna in the data ingestion step


**Deduplication: Identify and remove duplicate transaction records.**


In [14]:
transactions_df_distinct = transactions_df.dropDuplicates()

In [15]:
print(f"Duplicate Entry = {transactions_df.count() - transactions_df_distinct.count()}")

Duplicate Entry = 3


**Standardization: Normalize inconsistent category values and cast columns to correct numeric/timestamp types.**<br>
We have already done this


In [16]:
products_df.printSchema()

root
 |-- product_id: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- brand: string (nullable = false)



In [17]:
products_df.show()

+----------+--------------------+-----------+
|product_id|        product_name|      brand|
+----------+--------------------+-----------+
|   PROD001|Wireless Noise Ca...|  TechSound|
|   PROD002|   Bluetooth Speaker|   AudioMax|
|   PROD003|Designer Leather ...| FashionHub|
|   PROD004|     Garden Tool Set|  GreenLife|
|   PROD005|Python Programmin...| BookMaster|
|   PROD006| 4K Smart TV 55 inch| VisionTech|
|   PROD007|Casual Cotton T-S...|  StyleWear|
|   PROD008|       Gaming Laptop|    TechPro|
|   PROD009|Data Science Hand...| LearnPress|
|   PROD010|Indoor Plant Pot Set|  HomeDecor|
|   PROD011|   Tennis Racket Pro|   SportMax|
|   PROD012| Digital Camera DSLR|  PhotoShot|
|   PROD013|Summer Dress Coll...|TrendyStyle|
|   PROD014|Classic Literatur...|  BookWorld|
|   PROD015|      Tablet 10 inch|   TechWave|
|   PROD016| Running Shoes Elite|  ActiveFit|
|   PROD017|     Kitchen Blender| CookMaster|
|   PROD018|        Evening Gown|ElegantWear|
|   PROD019|   Smartphone 5G Pro| 

In [18]:
transactions_df_distinct.printSchema()

root
 |-- transaction_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- product_id: string (nullable = false)
 |-- category: string (nullable = false)
 |-- price: float (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- transaction_timestamp: timestamp (nullable = false)
 |-- platform: string (nullable = false)
 |-- country: string (nullable = false)



In [19]:
transactions_df_distinct.show()

+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|     category|  price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|        TXN042|USER1042|   PROD042|  ELECTRONICS|1099.99|       1|  2024-01-19 09:15:30|     Web| France|
|        TXN128|USER1128|   PROD028|       Sports| 189.99|       1|  2024-01-27 15:10:45|     Web|Germany|
|        TXN112|USER1112|   PROD012|  Electronics|  799.0|       1|  2024-01-26 09:15:30|     Web| France|
|        TXN187|USER1187|   PROD087|      Fashion|  115.0|       1|  2024-02-02 14:22:35|  Mobile| France|
|        TXN076|USER1076|   PROD076|  Electronics|1999.99|       1|  2024-01-22 13:15:20|     Web|    USA|
|        TXN129|USER1129|   PROD029|  Electronics|  850.0|       1|  2024-01-27 16:30:15|  Mobile| Canada|
|        TXN137|USER1137|   PROD037|H

In [20]:
transactions_df_distinct.groupBy('country').agg(count("transaction_id").alias("total_transactions")).show()

+-------+------------------+
|country|total_transactions|
+-------+------------------+
|Germany|                38|
| France|                38|
|    USA|                42|
|     UK|                39|
| Canada|                37|
+-------+------------------+



# 4. Partitioning & Performance


**Inspect and modify the number of partitions to optimize parallel processing.**



In [21]:
products_df.rdd.getNumPartitions()

1

In [22]:
transactions_df_distinct.rdd.getNumPartitions()

1

**Use coalesce() or repartition() appropriately to reduce partitions before writing.**

In [23]:
transaction_df_partitioned = transactions_df_distinct.repartition(4)

In [24]:
transaction_df_partitioned.rdd.getNumPartitions()

4

**Implement caching or persistence only for DataFrames reused in multiple actions.**

In [25]:
transaction_df_partitioned.cache()

DataFrame[transaction_id: string, user_id: string, product_id: string, category: string, price: float, quantity: int, transaction_timestamp: timestamp, platform: string, country: string]

# 5. Business Analytics & Aggregations


**Total Revenue: Calculated per country.**

In [26]:
transaction_df_partitioned.show()

+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|     category|  price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|        TXN042|USER1042|   PROD042|  ELECTRONICS|1099.99|       1|  2024-01-19 09:15:30|     Web| France|
|        TXN131|USER1131|   PROD031|Home & garden|  325.0|       2|  2024-01-28 08:30:45|  Mobile|    USA|
|        TXN167|USER1167|   PROD067|Home & Garden| 245.99|       2|  2024-01-31 14:22:35|  Mobile| France|
|        TXN059|USER1059|   PROD059|  electronics|  775.0|       1|  2024-01-20 16:30:15|  Mobile| Canada|
|        TXN063|USER1063|   PROD063|      fashion|  155.0|       2|  2024-01-21 10:22:15|  Mobile|Germany|
|        TXN001|USER1001|   PROD001|  Electronics| 599.99|       1|  2024-01-15 10:30:45|  Mobile|    USA|
|        TXN208|USER1208|   PROD008| 

In [27]:
transaction_df_partitioned.groupBy('country').agg(
    round(
        sum(
            col('price')*col('quantity')
            )
        ,2).alias("total_revenue")
    ).show()

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|Germany|     10898.77|
| France|     23598.71|
|    USA|     33045.79|
|     UK|     16451.73|
| Canada|     24987.34|
+-------+-------------+



**Top Performers: Identify the top 5 products by total revenue.**

In [28]:
transaction_df_partitioned.groupBy('product_id').agg(
    round(
        sum(
            col('price')*col('quantity')
            )
        ,2).alias("total_revenue")
    ).orderBy(['total_revenue'],ascending = False).show(5)

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|   PROD076|      3999.98|
|   PROD046|      3599.98|
|   PROD092|      3399.98|
|   PROD032|      3199.98|
|   PROD086|      3199.98|
+----------+-------------+
only showing top 5 rows


**Platform Metrics: Average order value per platform.**

In [29]:
transaction_df_partitioned.groupBy('platform').agg(
    round(
        avg(
            col('price')*col('quantity')
            )
        ,2).alias("avg_revenue")
    ).orderBy(['avg_revenue'],ascending = False).show()

+--------+-----------+
|platform|avg_revenue|
+--------+-----------+
|     Web|     692.47|
|  Mobile|     425.55|
+--------+-----------+



**Trends: Daily transaction count trends.**

In [30]:

transaction_df_partitioned = transaction_df_partitioned.withColumn(
    "transaction_date",
    to_date(col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss")
)


In [31]:

daily_trend = (
    transaction_df_partitioned.groupBy("transaction_date")
      .agg(count("*").alias("transaction_date"))
      .orderBy("transaction_date")
)

In [32]:
daily_trend.show()

+----------------+----------------+
|transaction_date|transaction_date|
+----------------+----------------+
|      2024-01-15|               7|
|      2024-01-16|               9|
|      2024-01-17|               9|
|      2024-01-18|              10|
|      2024-01-19|               9|
|      2024-01-20|              10|
|      2024-01-21|              10|
|      2024-01-22|               8|
|      2024-01-23|              10|
|      2024-01-24|              10|
|      2024-01-25|               9|
|      2024-01-26|              10|
|      2024-01-27|              10|
|      2024-01-28|              10|
|      2024-01-29|              10|
|      2024-01-30|              10|
|      2024-01-31|              10|
|      2024-02-01|              10|
|      2024-02-02|              10|
|      2024-02-03|              10|
+----------------+----------------+
only showing top 20 rows


# 6. Data Integration & Advanced Features


**Join Operations: Join transactional data with a reference dataset (Product name, Brand) using optimization techniques like Broadcast Joins.**

In [33]:
products_df.show(3)

+----------+--------------------+----------+
|product_id|        product_name|     brand|
+----------+--------------------+----------+
|   PROD001|Wireless Noise Ca...| TechSound|
|   PROD002|   Bluetooth Speaker|  AudioMax|
|   PROD003|Designer Leather ...|FashionHub|
+----------+--------------------+----------+
only showing top 3 rows


In [34]:
transaction_df_partitioned.show(3)

+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+----------------+
|transaction_id| user_id|product_id|     category|  price|quantity|transaction_timestamp|platform|country|transaction_date|
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+----------------+
|        TXN042|USER1042|   PROD042|  ELECTRONICS|1099.99|       1|  2024-01-19 09:15:30|     Web| France|      2024-01-19|
|        TXN131|USER1131|   PROD031|Home & garden|  325.0|       2|  2024-01-28 08:30:45|  Mobile|    USA|      2024-01-28|
|        TXN167|USER1167|   PROD067|Home & Garden| 245.99|       2|  2024-01-31 14:22:35|  Mobile| France|      2024-01-31|
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+----------------+
only showing top 3 rows


In [35]:
transaction_df_partitioned.join(broadcast(products_df),on='product_id',how='left').show(3)

+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+----------------+-------------------+----------+
|product_id|transaction_id| user_id|     category|  price|quantity|transaction_timestamp|platform|country|transaction_date|       product_name|     brand|
+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+----------------+-------------------+----------+
|   PROD042|        TXN042|USER1042|  ELECTRONICS|1099.99|       1|  2024-01-19 09:15:30|     Web| France|      2024-01-19|   Portable SSD 2TB| DataStore|
|   PROD031|        TXN131|USER1131|Home & garden|  325.0|       2|  2024-01-28 08:30:45|  Mobile|    USA|      2024-01-28|Wall Art Canvas Set|   HomeArt|
|   PROD067|        TXN167|USER1167|Home & Garden| 245.99|       2|  2024-01-31 14:22:35|  Mobile| France|      2024-01-31|   Towel Set Cotton|BathLuxury|
+----------+--------------+--------+-------------+-------+--------+---

In [36]:
transaction_df_partitioned.join(broadcast(products_df),on='product_id',how='left').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#282, transaction_id#280, user_id#281, category#283, price#284, quantity#285, transaction_timestamp#286, platform#287, country#288, transaction_date#1375, product_name#269, brand#270]
   +- BroadcastHashJoin [product_id#282], [product_id#268], LeftOuter, BuildRight, false
      :- Project [transaction_id#280, user_id#281, product_id#282, category#283, price#284, quantity#285, transaction_timestamp#286, platform#287, country#288, cast(gettimestamp(transaction_timestamp#286, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_date, Some(Asia/Calcutta), true) as date) AS transaction_date#1375]
      :  +- InMemoryTableScan [category#283, country#288, platform#287, price#284, product_id#282, quantity#285, transaction_id#280, transaction_timestamp#286, user_id#281]
      :        +- InMemoryRelation [transaction_id#280, user_id#281, product_id#282, category#283, price#284, quantity#285, transaction_timestamp#286, platform#

**Tracking: Utilize Accumulators to track specific metrics (e.g., total invalid records) across the cluster.**

In [37]:

invalid_count = spark.sparkContext.accumulator(0)

In [38]:
def validate_record(row):
    if any(value is None for value in row):
        invalid_count.add(1)
    return row

In [39]:

validated_df = invalid_transactions.rdd.map(validate_record).toDF(invalid_transactions.schema)


In [40]:
validated_df.show()

print("Invalid Records:", invalid_count.value)


+--------------+--------+----------+-------------+------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|     category| price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-------------+------+--------+---------------------+--------+-------+
|        TXN205|    NULL|   PROD005|        Books| 25.99|       1|  2024-02-04 12:30:45|  Mobile|     UK|
|        TXN043|USER1043|   PROD043|      Fashion|  NULL|       2|  2024-01-19 10:22:15|  Mobile|Germany|
|        TXN206|USER1206|   PROD006|  ELECTRONICS|   abc|       1|  2024-02-04 13:15:20|     Web|    USA|
|        TXN008|USER1008|      NULL|  Electronics|399.99|       1|  2024-01-15 17:05:30|     Web| Canada|
|          NULL|    NULL|      NULL|         NULL|  NULL|    NULL|                 NULL|    NULL|   NULL|
|        TXN009|    NULL|   PROD009|        Books| 15.99|       1|  2024-01-15 18:12:22|  Mobile|     UK|
|        TXN005|USER1005|   PROD005|        Bo

# 7. Output Management

In [43]:
products_df.show(3)

+----------+--------------------+----------+
|product_id|        product_name|     brand|
+----------+--------------------+----------+
|   PROD001|Wireless Noise Ca...| TechSound|
|   PROD002|   Bluetooth Speaker|  AudioMax|
|   PROD003|Designer Leather ...|FashionHub|
+----------+--------------------+----------+
only showing top 3 rows


In [44]:
products_df.printSchema()

root
 |-- product_id: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- brand: string (nullable = false)



In [42]:
transactions_df_distinct.show(3)

+--------------+--------+----------+-----------+-------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|   category|  price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-----------+-------+--------+---------------------+--------+-------+
|        TXN042|USER1042|   PROD042|ELECTRONICS|1099.99|       1|  2024-01-19 09:15:30|     Web| France|
|        TXN128|USER1128|   PROD028|     Sports| 189.99|       1|  2024-01-27 15:10:45|     Web|Germany|
|        TXN112|USER1112|   PROD012|Electronics|  799.0|       1|  2024-01-26 09:15:30|     Web| France|
+--------------+--------+----------+-----------+-------+--------+---------------------+--------+-------+
only showing top 3 rows


In [45]:
transactions_df_distinct.printSchema()

root
 |-- transaction_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- product_id: string (nullable = false)
 |-- category: string (nullable = false)
 |-- price: float (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- transaction_timestamp: timestamp (nullable = false)
 |-- platform: string (nullable = false)
 |-- country: string (nullable = false)



In [46]:
final_df = transactions_df_distinct.join(
    products_df,
    on="product_id",
    how="left"
)

In [51]:
final_df.write \
    .mode("overwrite") \
    .partitionBy("category") \
    .parquet("output/")

Py4JJavaError: An error occurred while calling o287.parquet.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.